In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

from src.utils import (
    get_args,
    set_seed,
    get_datesets_and_loaders,
    get_trained_VAE,
    get_trained_VAE_with_domain_classifier,
    get_trained_classifier,
    get_trained_classifier_Base,
    test_model,
    prepare_report,
    run_all_senario,
)
from src.tupl import run_tupl
from src.our_tupl import GENERATION_POLICIES, run_m1_tupl, run_all_senario_m1_tupl
from src.vista_gzsda import run_vista
from src.utils_report import METHOD_ORDER, results_to_dataframe, times_to_dataframe

/home/asad/workspace/DomainProject/changeDomain/notebooks/effective-gzsda/gzsda/src/utils.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy
/home/asad/workspace/anaconda3/envs/asad/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
STYLE_LABELS = [
    "angry", "childlike", "depressed", "neutral",
    "old", "proud", "strutting",
]
DOMAIN_SET = [f"all_but_{s}" for s in STYLE_LABELS] + STYLE_LABELS
PAIRS = [(i, i + len(STYLE_LABELS)) for i in range(len(STYLE_LABELS))]
DATA_DIR = "./data/ActionStyleDataset_v2/"
DATASET_DETAILS = {
    "prefix": "ActionStyle-",
    "suffix": "-clip.mat",
    "resnet_feature": "clip_features",
    "split_file_name": "instanceSplit_actionStyle_v2_unseen2.mat",
}
NUM_LABELS = 5

print("DOMAIN_SET", DOMAIN_SET)
print("PAIRS", [(DOMAIN_SET[s], DOMAIN_SET[t]) for s, t in PAIRS])

DOMAIN_SET ['all_but_angry', 'all_but_childlike', 'all_but_depressed', 'all_but_neutral', 'all_but_old', 'all_but_proud', 'all_but_strutting', 'angry', 'childlike', 'depressed', 'neutral', 'old', 'proud', 'strutting']
PAIRS [('all_but_angry', 'angry'), ('all_but_childlike', 'childlike'), ('all_but_depressed', 'depressed'), ('all_but_neutral', 'neutral'), ('all_but_old', 'old'), ('all_but_proud', 'proud'), ('all_but_strutting', 'strutting')]


In [4]:
import json
import time
from pathlib import Path

RESULT_OBJ_PATH = "./result/json/actionStyle_v2.json"
RESULT_CSV_PATH = "./result/csv/actionStyle_v2.csv"
RESULT_TIME_PATH = "./result/times/raw_time_actionStyle_v2.json"
RESULT_TIME_CSV_PATH = "./result/times/result_time_actionStyle_v2.csv"
path = Path(RESULT_OBJ_PATH)
time_path = Path(RESULT_TIME_PATH)

if path.exists():
    with path.open("r", encoding="utf-8") as f:
        result = json.load(f)
else:
    result = {}

if time_path.exists():
    with time_path.open("r", encoding="utf-8") as f:
        result_time = json.load(f)
else:
    result_time = {}

def save_results():
    time_path.parent.mkdir(parents=True, exist_ok=True)
    with open(RESULT_OBJ_PATH, "w") as f:
        json.dump(result, f, indent=2)
    with open(RESULT_TIME_PATH, "w") as f:
        json.dump(result_time, f, indent=2)

result.keys(), result_time.keys()


(dict_keys(['base', 'CCVAE', 'our0', 'our_GRE', 'TUPL', 'our_TUPL_real_plus_src2tgt', 'our_TUPL_real_plus_src2tgt_unseen', 'our_TUPL_interp_src2tgt', 'VisTA']),
 dict_keys(['base', 'CCVAE', 'our0', 'our_GRE', 'TUPL', 'our_TUPL_real_plus_src2tgt', 'our_TUPL_real_plus_src2tgt_unseen', 'our_TUPL_interp_src2tgt', 'VisTA']))

In [5]:
base = "base"
CCVAE = "CCVAE"
our0 = "our0"
our_GRE = "our_GRE"
tupl = "TUPL"
our_tupl = "our_TUPL"
vista = "VisTA"

# clear last result
# _, _ = result.pop(base, None), result_time.pop(base, None)
# _, _ = result.pop(CCVAE, None), result_time.pop(CCVAE, None)
# _, _ = result.pop(our0, None), result_time.pop(our0, None)
# _, _ = result.pop(our_GRE, None), result_time.pop(our_GRE, None)
# _, _ = result.pop(tupl, None), result_time.pop(tupl, None)
# _, _ = result.pop(vista, None), result_time.pop(vista, None)
# for k in [f"{our_tupl}_{p}" for p in GENERATION_POLICIES]:
#     _, _ = result.pop(k, None), result_time.pop(k, None)


## Base


In [6]:
def main_base(args):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    classifier = get_trained_classifier_Base(
        data_loaders=data_loaders,
        NUM_LABELS=NUM_LABELS,
        device=device,
        input_dim=512)

    return test_model(classifier, datasets["test"], data_loaders["test"], device)


In [7]:
if base not in result or base not in result_time:
    start = time.time()
    result[base] = run_all_senario(main_base, DOMAIN_SET, input_dim=512, num_trial=6, pairs=PAIRS)
    result_time[base] = time.time() - start
    save_results()


# GZSDA


In [8]:
def main_gzsda(args):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        input_dim=512)

    return test_model(classifier, datasets["test"], data_loaders["test"], device)


In [9]:
if CCVAE not in result or CCVAE not in result_time:
    start = time.time()
    result[CCVAE] = run_all_senario(main_gzsda, DOMAIN_SET, input_dim=512, num_trial=6, pairs=PAIRS)
    result_time[CCVAE] = time.time() - start
    save_results()


## m0


In [10]:
def main_m0(args):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30,
        input_dim=512)

    return test_model(classifier, datasets["test"], data_loaders["test"], device)


In [11]:
if our0 not in result or our0 not in result_time:
    start = time.time()
    result[our0] = run_all_senario(main_m0, DOMAIN_SET, input_dim=512, num_trial=6, pairs=PAIRS)
    result_time[our0] = time.time() - start
    save_results()


## m1: seperate after encoder


In [12]:
def main_m1(args):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE_with_domain_classifier(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30,
        input_dim=512)

    return test_model(classifier, datasets["test"], data_loaders["test"], device)


In [13]:
if our_GRE not in result or our_GRE not in result_time:
    start = time.time()
    result[our_GRE] = run_all_senario(main_m1, DOMAIN_SET, input_dim=512, num_trial=6, pairs=PAIRS)
    result_time[our_GRE] = time.time() - start
    save_results()


## TUPL

Leave-one-style-out: source is `all_but_X` (six styles concatenated), target is `X`.

In [14]:
def main_tupl(args):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    acc_s, acc_u, h = run_tupl(
        data_root="./data/",
        dataset="actionstyle_v2",
        source=args.sourceDomainIndex,
        target=args.targetDomainIndex,
        trial=args.trialIndex,
        seed=args.seed,
        device=device,
        quiet=True,
        return_model=False,
    )
    print("seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}".format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0

In [15]:
if tupl not in result or tupl not in result_time:
    start = time.time()
    result[tupl] = run_all_senario(
        main_tupl, DOMAIN_SET, input_dim=512, num_trial=6, pairs=PAIRS
    )
    result_time[tupl] = time.time() - start
    save_results()


## our_TUPL: m1 VAE + TUPL


In [16]:
def main_m1_tupl(args, policy="real_plus_src2tgt"):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    acc_s, acc_u, h = run_m1_tupl(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS,
        policy=policy,
        device=device,
        quiet=True,
    )
    print("seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}".format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0


In [17]:
for p in GENERATION_POLICIES:
    key = f"{our_tupl}_{p}"
    if key not in result or key not in result_time:
        start = time.time()
        result.update(run_all_senario_m1_tupl(
            DOMAIN_SET=DOMAIN_SET,
            DATA_DIR=DATA_DIR,
            DATASET_DETAILS=DATASET_DETAILS,
            policies=[p],
            input_dim=512,
            num_trial=6,
            pairs=PAIRS,
        ))
        result_time[key] = time.time() - start
        save_results()


## VisTA

Feature-space VisTA on MotionCLIP embeddings (no images, no Grad-CAM VAC).

In [18]:
def main_vista(args):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    return run_vista(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS,
        device=device,
        quiet=True,
    )

In [19]:
if vista not in result or vista not in result_time:
    start = time.time()
    result[vista] = run_all_senario(
        main_vista, DOMAIN_SET, input_dim=512, num_trial=6, pairs=PAIRS
    )
    result_time[vista] = time.time() - start
    save_results()


## Merge results

In [20]:
save_results()


In [21]:
# ignore our0
_, _ = result.pop(our0, None), result_time.pop(our0, None)
_, _ = result.pop("our_TUPL_real_plus_src2tgt", None), result_time.pop("our_TUPL_real_plus_src2tgt", None)
_, _ = result.pop("our_TUPL_real_plus_src2tgt_unseen", None), result_time.pop("our_TUPL_real_plus_src2tgt_unseen", None)
# _, _ = result.pop("our_TUPL_interp_src2tgt", None), result_time.pop("our_TUPL_interp_src2tgt", None)


In [22]:
n_senario = len(PAIRS)
n_trial = 6

df_time = times_to_dataframe(result_time, n_senario=n_senario, n_trial=n_trial)
df_time.to_csv(RESULT_TIME_CSV_PATH, index=False)
# df_time


In [23]:
df = results_to_dataframe(result, METHOD_ORDER)
df


,domain,method,seen,unseen,H-mean
0,all_but_angry -> angry,base,100.00 ± 0.00,91.67 ± 2.58,95.56 ± 1.40
1,all_but_angry -> angry,VisTA,70.03 ± 4.53,86.01 ± 4.18,77.14 ± 4.37
2,all_but_angry -> angry,CCVAE,100.00 ± 0.00,85.52 ± 4.35,91.90 ± 2.53
3,all_but_angry -> angry,our_GRE,99.07 ± 0.93,94.10 ± 1.65,96.48 ± 1.00
4,all_but_angry -> angry,TUPL,94.79 ± 4.09,45.24 ± 5.09,60.57 ± 4.98
5,all_but_angry -> angry,our_TUPL_interp_src2tgt,95.62 ± 2.88,46.64 ± 8.96,59.31 ± 10.18
6,all_but_childlike -> childlike,base,95.83 ± 2.08,91.18 ± 2.11,93.28 ± 1.12
7,all_but_childlike -> childlike,VisTA,74.65 ± 10.21,45.48 ± 11.73,47.48 ± 9.43
8,all_but_childlike -> childlike,CCVAE,94.44 ± 2.73,84.33 ± 3.56,88.76 ± 2.05
9,all_but_childlike -> childlike,our_GRE,93.06 ± 3.79,95.36 ± 0.74,94.01 ± 2.16


In [24]:
Path(RESULT_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RESULT_CSV_PATH, index=False)